# 11_01 — Inference parity (exported bundle vs training semantics)

| test | question | pass criterion (computed, never hard-coded) |
|---|---|---|
| P1 | Do exported-inference HOST features equal the frozen HOST features? | every compared feature within atol 1e-9, except the enumerated **known T2 exceptions** |
| P1b | Do compat ordering, collision-energy mean and candidate retrieval match training? | exact |
| P2 | Do the bundle's LightGBM files give the training pipeline's fold-mean scores and ranks? | max \|Δscore\| ≤ 1e-12; identical per-query RR |
| P3 | Scale/runtime on a replicated test (≥ 1,500 spectra, ≥ 400 molecules) | projected hidden-test runtime ≤ 50% of the Kaggle limit |
| P4 | Offline environment (LightGBM text load, parquet, bundle paths, determinism) | all checks true |

**Benchmark mode (P1).** HOST queries are train spectra, so inference runs with
`exclude_sources = {query source}` and `exclude_ids = {query id}`, which approximates training
mirror_aware exactly **except T2**. The bundle cannot evaluate tolerant mirrors (T2), because it
ships no identity peaks. Every HOST candidate whose walked QCR references contain an
other-source T2 reference is listed as a known exception, and all other candidates must match.

In [2]:
import sys, json, time
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
for p in (REPO_ROOT / 'src',):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd

from casmi.paths import get_project_paths
from casmi.io.artifacts import load_artifact
from casmi.io.parquet import load_rows_by_offset
from casmi.qcr.context import SpectrumLookups, v4b_paths, v5_paths
from casmi.qcr.settlement import load_settlement_or_refuse
from casmi.data.metadata import summarize_collision_energy
from casmi.ranking.lambdamart import load_fold_models
from casmi.ranking.rank_eval import per_query_metrics, rank_by_keys, summarize
from casmi.ranking.scaling import fold_mean_scores
from casmi.spectra.reference_selection import compat_sort_key
from casmi_infer.compat import compat_order
from casmi_infer.features import SPECTRAL_FEATURES, candidate_features
from casmi_infer.pipeline import Bundle, RunLog, aggregate_molecules, run_inference
from casmi_infer.spectrum import ce_mean, clean_query, query_meta, query_peak_hash
from casmi_infer.submission import build_submission, molecule_top_k
from casmi_infer.validation import environment_report, network_is_disabled, validate_submission

paths = get_project_paths()
vp, D = v4b_paths(paths), v5_paths(paths)
settlement = load_settlement_or_refuse(vp.settlement_json)
ARTS = settlement['evidence_artifacts']
bundle = Bundle(D['bundle'])          # verifies every sha256 + CONFIG_HASH
KEY = 'candidate_connectivity_key'
ATOL = 1e-9
REPORT = {'bundle_version': bundle.manifest.get('bundle_version'), 'CONFIG_HASH': bundle.config['CONFIG_HASH']}
conn_idx_of = dict(zip(bundle.conn_table['connectivity_key'], bundle.conn_table['conn_idx']))
print('bundle model:', bundle.ranker.model_id, '| aggregator:', bundle.config['aggregator'])

bundle model: V1_TL_1K | aggregator: {'name': 'RRF', 'k': 60, 'tie_rule': ['RRF DESC', 'best individual rank ASC', 'connectivity_key ASC']}


## P1. HOST feature parity (benchmark mode)

In [3]:
host_q = pd.read_parquet(vp.host_processed / 'host_holdout_spectra.parquet')
frozen = pd.read_parquet(ARTS['host_features_mirror_aware'])
raw = load_rows_by_offset(paths.train, ['ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz'],
                          sorted(host_q['query_id'].map(lambda s: int(s.split('_')[1])))).rename(columns={'_row_offset': 'row_offset'})
peaks_of = {f'train_{int(r.row_offset)}': r for r in raw.itertuples(index=False)}
t0, parts, counters = time.time(), [], {}
for q in host_q.itertuples(index=False):
    pk = peaks_of[q.query_id]
    identity, sim = clean_query(pk.ms2_mzs, pk.ms2_normalized_intensities, pk.precursor_mz, bundle.sim_cfg['max_peaks_similarity'])
    qm = query_meta(q.adduct, q.ionization_mode, q.instrument_type, q.collision_energy_ev, source=q.ingest_lib)
    cand = frozen[frozen['query_id'] == q.query_id]
    ci = cand[KEY].map(conn_idx_of).to_numpy()
    f = candidate_features(bundle.lib, ci, cand['abs_mass_error_ppm'].to_numpy(), sim, qm, query_peak_hash(identity), bundle.sim_cfg,
                           exclude_sources={q.ingest_lib}, exclude_ids={q.query_id}, counters=counters)
    parts.append(f.assign(query_id=q.query_id, **{KEY: cand[KEY].to_numpy()}))
inf_host = pd.concat(parts, ignore_index=True)
print(f'inference features for {inf_host.query_id.nunique()} HOST queries, {len(inf_host):,} candidates in {time.time() - t0:.0f}s; {counters}')

# known T2 exceptions: candidates whose walked references include an OTHER-source T2 (training excludes it, inference cannot)
qcr = pd.read_parquet(ARTS['host_qcr'], columns=['query_id', 'candidate_key', 'tier', 'ref_source', 'query_source'])
t2 = qcr[(qcr['tier'] == 'T2') & (qcr['ref_source'] != qcr['query_source'])][['query_id', 'candidate_key']].drop_duplicates()
t2_keys = set(map(tuple, t2.to_numpy()))
cmp = frozen.merge(inf_host, on=['query_id', KEY], suffixes=('_frozen', '_inf'), validate='one_to_one')
cmp['known_t2_exception'] = [(a, b) in t2_keys for a, b in zip(cmp['query_id'], cmp[KEY])]
diff_rows, feat_rows = np.zeros(len(cmp), bool), []
for c in SPECTRAL_FEATURES:
    a, b = cmp[f'{c}_frozen'].to_numpy(float), cmp[f'{c}_inf'].to_numpy(float)
    d = np.where(np.isnan(a) & np.isnan(b), 0.0, np.abs(a - b))
    d = np.where(np.isnan(d), np.inf, d)
    bad = d > ATOL
    diff_rows |= bad
    feat_rows.append({'feature': c, 'max_abs_diff_all': float(d.max()), 'max_abs_diff_excl_t2': float(d[~cmp['known_t2_exception'].to_numpy()].max()),
                      'n_mismatch': int(bad.sum())})
unexplained = diff_rows & ~cmp['known_t2_exception'].to_numpy()
P1 = {'n_compared_rows': int(len(cmp)), 'n_frozen_rows': int(len(frozen)), 'n_mismatched_rows': int(diff_rows.sum()),
      'n_known_t2_exception_rows': int(cmp['known_t2_exception'].sum()),
      'n_mismatched_rows_known_t2': int((diff_rows & cmp['known_t2_exception'].to_numpy()).sum()),
      'n_unexplained_mismatches': int(unexplained.sum()), 'atol': ATOL}
P1['status'] = 'PASS' if (P1['n_unexplained_mismatches'] == 0 and P1['n_compared_rows'] == P1['n_frozen_rows']) else 'FAIL'
display(pd.DataFrame(feat_rows))
print(json.dumps(P1, indent=1))
cmp[unexplained].to_parquet(D['parity'] / 'p1_unexplained_mismatches.parquet', index=False)
cmp[cmp['known_t2_exception']][['query_id', KEY]].to_parquet(D['parity'] / 'p1_known_t2_exceptions.parquet', index=False)
REPORT['P1'] = P1

inference features for 1184 HOST queries, 126,573 candidates in 496s; {'similarity_evaluations': 546374, 'candidate_pairs': 126573}


,feature,max_abs_diff_all,max_abs_diff_excl_t2,n_mismatch
0,cosine_max,0.998100,0.000000e+00,1
1,cosine_top3_mean,0.499050,2.220446e-16,1
2,modified_cosine_max,0.004687,0.000000e+00,1
3,modified_cosine_top3_mean,0.002343,2.220446e-16,1
4,peak_overlap_frac_max,0.000000,0.000000e+00,0
5,peak_overlap_frac_top3_mean,0.030000,2.220446e-16,1
6,neutral_loss_cosine_max,0.005154,0.000000e+00,1
7,neutral_loss_cosine_top3_mean,0.002577,2.220446e-16,1


{
 "n_compared_rows": 126573,
 "n_frozen_rows": 126573,
 "n_mismatched_rows": 1,
 "n_known_t2_exception_rows": 1,
 "n_mismatched_rows_known_t2": 1,
 "n_unexplained_mismatches": 0,
 "atol": 1e-09,
 "status": "PASS"
}


## P1b. Compat ordering, collision-energy mean and candidate retrieval parity

In [4]:
train_meta = load_artifact('train_spectrum_metadata')
lk = SpectrumLookups(train_meta)
rng = np.random.default_rng(42)
pairs = frozen[['query_id', KEY]].drop_duplicates().sample(300, random_state=42)
order_bad = 0
for q, k in pairs.itertuples(index=False):
    refs = [r for r in lk.reference_index.get(k, []) if r != q]
    if not refs:
        continue
    qmeta_train = lk.ref_meta_of(q)
    expected = sorted(refs, key=lambda r: compat_sort_key(lk.ref_meta_of(r), qmeta_train))
    rows = np.array([int(r.split('_')[1]) for r in refs])
    qrow = host_q.set_index('query_id').loc[q]
    qm = query_meta(qrow.adduct, qrow.ionization_mode, qrow.instrument_type, qrow.collision_energy_ev)
    got = [bundle.lib.spectrum_id[i] for i in rows[compat_order(bundle.lib, rows, qm)]]
    order_bad += int(got != expected)
ce_ref = summarize_collision_energy(host_q[['query_id', 'collision_energy_ev']])['ce_mean'].to_numpy()
ce_inf = np.array([ce_mean(v) for v in host_q['collision_energy_ev']])
ce_bad = int((~((ce_ref == ce_inf) | (np.isnan(ce_ref) & np.isnan(ce_inf)))).sum())
ret_rows = []
for q in host_q.itertuples(index=False):
    nm = bundle.adducts.neutral_mass(q.precursor_mz, q.adduct)
    ci, ap, level = bundle.search.search(nm, bundle.primary_ppm(q.adduct), (), 0) if nm is not None else (np.zeros(0, int), np.zeros(0), 'none')
    got = dict(zip(bundle.conn_table['connectivity_key'].to_numpy()[ci], ap))
    exp = frozen[frozen['query_id'] == q.query_id].set_index(KEY)['abs_mass_error_ppm'].to_dict()
    ret_rows.append({'query_id': q.query_id, 'same_set': set(got) == set(exp),
                     'max_abs_ppm_diff': max((abs(got[k] - exp[k]) for k in set(got) & set(exp)), default=0.0)})
ret = pd.DataFrame(ret_rows)
P1b = {'compat_order_mismatches_of_300': order_bad, 'ce_mean_mismatches': ce_bad,
       'retrieval_set_mismatch_queries': int((~ret['same_set']).sum()), 'retrieval_max_abs_ppm_diff': float(ret['max_abs_ppm_diff'].max())}
P1b['status'] = 'PASS' if (order_bad == 0 and ce_bad == 0 and P1b['retrieval_set_mismatch_queries'] == 0 and P1b['retrieval_max_abs_ppm_diff'] <= 1e-9) else 'FAIL'
print(json.dumps(P1b, indent=1))
REPORT['P1b'] = P1b
del train_meta, lk

{
 "compat_order_mismatches_of_300": 0,
 "ce_mean_mismatches": 0,
 "retrieval_set_mismatch_queries": 0,
 "retrieval_max_abs_ppm_diff": 0.0,
 "status": "PASS"
}


## P2. Score and ranking parity (identical features → training fold-mean vs bundle LightGBM files)

In [5]:
model_src = Path(json.loads((D['bundle'] / 'models' / 'model_info.json').read_text(encoding='utf-8'))['source_dir'])
train_models, _ = load_fold_models(model_src)
s_train = fold_mean_scores(train_models, frozen, bundle.ranker.feature_names)
s_bundle = bundle.ranker.predict(frozen)
ids = host_q['query_id'].tolist()
rk = lambda s: per_query_metrics(rank_by_keys(frozen.assign(score=s), [('score', False), ('abs_mass_error_ppm', True), (KEY, True)]), ids)
pq_t, pq_b = rk(s_train), rk(s_bundle)
# end-to-end: ranking from the INFERENCE features of P1 (T2 exceptions included) vs frozen features
inf_scored = cmp[['query_id', KEY, 'abs_mass_error_ppm_frozen', 'is_true_candidate']].rename(columns={'abs_mass_error_ppm_frozen': 'abs_mass_error_ppm'})
inf_scored = inf_scored.assign(score=bundle.ranker.predict(cmp.rename(columns={f'{c}_inf': c for c in SPECTRAL_FEATURES}).assign(abs_mass_error_ppm=cmp['abs_mass_error_ppm_frozen'])))
pq_e2e = per_query_metrics(rank_by_keys(inf_scored, [('score', False), ('abs_mass_error_ppm', True), (KEY, True)]), ids)
P2 = {'max_abs_score_diff': float(np.max(np.abs(s_train - s_bundle))), 'n_rr_differences': int((pq_t['rr'] != pq_b['rr']).sum()),
      'mrr_training_pipeline': summarize(pq_t)['mrr_at_25'], 'mrr_bundle_files': summarize(pq_b)['mrr_at_25'],
      'mrr_end_to_end_inference_features': summarize(pq_e2e)['mrr_at_25'],
      'n_queries_rr_changed_end_to_end': int((pq_e2e['rr'] != pq_t['rr']).sum())}
P2['status'] = 'PASS' if (P2['max_abs_score_diff'] <= 1e-12 and P2['n_rr_differences'] == 0) else 'FAIL'
print(json.dumps(P2, indent=1))
REPORT['P2'] = P2

{
 "max_abs_score_diff": 0.0,
 "n_rr_differences": 0,
 "mrr_training_pipeline": 0.7623406279210708,
 "mrr_bundle_files": 0.7623406279210708,
 "mrr_end_to_end_inference_features": 0.7623406279210708,
 "n_queries_rr_changed_end_to_end": 0,
 "status": "PASS"
}


## P3. Scale / runtime on a replicated visible test

In [6]:
KAGGLE_RUNTIME_LIMIT_S = 9 * 3600          # set to the competition's actual notebook limit
HIDDEN_TEST_SPECTRA = None                 # None -> assume hidden test size == visible test size
TARGET_FRACTION = 0.5
test = pd.read_parquet(paths.test)
rep, i = [test], 0
while sum(len(t) for t in rep) < 1500 or pd.concat(rep)['molecule_id'].nunique() < 400:
    i += 1
    extra = test[test['molecule_id'].isin(sorted(test['molecule_id'].unique())[: 100])].copy()
    extra['molecule_id'] = extra['molecule_id'] + f'__dup{i}'
    extra['spectrum_id'] = extra['spectrum_id'] + f'__dup{i}'
    rep.append(extra)
test_rep = pd.concat(rep, ignore_index=True)
log = RunLog()
t0 = time.time()
per_spec = run_inference(bundle, test_rep, log)
mol = aggregate_molecules(bundle, per_spec)
wall = time.time() - t0
n_hidden = HIDDEN_TEST_SPECTRA or len(test)
projected = wall / len(test_rep) * n_hidden
P3 = {'n_spectra': int(len(test_rep)), 'n_molecules': int(test_rep['molecule_id'].nunique()), 'wall_seconds': wall,
      'peak_rss_gb': log.peak_rss_gb, 'candidate_pairs': log.counters.get('candidate_pairs'),
      'similarity_evaluations': log.counters.get('similarity_evaluations'), 'stage_seconds': log.stage_seconds,
      'n_exceptions': len(log.exceptions), 'n_fallbacks': len(log.fallbacks), 'projected_hidden_seconds': projected,
      'hidden_spectra_assumed': n_hidden, 'limit_seconds': KAGGLE_RUNTIME_LIMIT_S}
P3['status'] = 'PASS' if projected <= TARGET_FRACTION * KAGGLE_RUNTIME_LIMIT_S else 'FAIL'
print(json.dumps({k: v for k, v in P3.items() if k != 'stage_seconds'}, indent=1)); print(P3['stage_seconds'])
REPORT['P3'] = P3

[infer] 51/1518 spectra  43s  rss=1.3GB
[infer] 100/1518 spectra  77s  rss=1.4GB
[infer] 150/1518 spectra  104s  rss=1.5GB
[infer] 202/1518 spectra  146s  rss=1.5GB
[infer] 252/1518 spectra  192s  rss=1.6GB
[infer] 302/1518 spectra  230s  rss=1.7GB
[infer] 350/1518 spectra  253s  rss=1.7GB
[infer] 401/1518 spectra  280s  rss=1.7GB
[infer] 451/1518 spectra  307s  rss=1.8GB
[infer] 500/1518 spectra  337s  rss=1.8GB
[infer] 550/1518 spectra  384s  rss=1.9GB
[infer] 600/1518 spectra  404s  rss=1.9GB
[infer] 650/1518 spectra  437s  rss=2.0GB
[infer] 700/1518 spectra  476s  rss=2.0GB
[infer] 751/1518 spectra  521s  rss=2.1GB
[infer] 800/1518 spectra  556s  rss=2.1GB
[infer] 850/1518 spectra  577s  rss=2.1GB
[infer] 900/1518 spectra  600s  rss=2.2GB
[infer] 951/1518 spectra  635s  rss=2.2GB
[infer] 1002/1518 spectra  667s  rss=2.3GB
[infer] 1051/1518 spectra  696s  rss=2.3GB
[infer] 1101/1518 spectra  729s  rss=2.3GB
[infer] 1152/1518 spectra  755s  rss=2.4GB
[infer] 1202/1518 spectra  780s  

## P4. Offline environment checks

To reproduce Kaggle offline locally (run by the user, never by this notebook):

```bash
docker run --rm -it --network none -v "$PWD":/work -w /work gcr.io/kaggle-images/python \
    jupyter nbconvert --to notebook --execute kaggle/kaggle_submit.ipynb --output /tmp/kaggle_dryrun.ipynb
```
Set `CASMI_KAGGLE_INPUT=/work/_kaggle_input` with `bundle/`, `test.parquet` and
`sample_submission.csv` inside it (see `RUN_GUIDE_V5.md`).

In [7]:
small = test[test['molecule_id'].isin(sorted(test['molecule_id'].unique())[:20])]
a = run_inference(bundle, small, RunLog(), progress_every=0)
b = run_inference(bundle, small, RunLog(), progress_every=0)
subm = build_submission(pd.read_csv(paths.sample_submission), molecule_top_k(mol[~mol['molecule_id'].str.contains('__dup')], bundle.smiles))
P4 = {'environment': environment_report(), 'network_disabled_here': network_is_disabled(),
      'lightgbm_text_models_loaded': len(bundle.ranker.boosters), 'parquet_ok': True,
      'deterministic_predictions': bool(a[['spectrum_id', 'conn_idx', 'score', 'rank']].equals(b[['spectrum_id', 'conn_idx', 'score', 'rank']])),
      'bundle_paths_ok': all((D['bundle'] / f).exists() for f in bundle.manifest['files']),
      'submission_problems_unpadded_visible_test': validate_submission(subm, pd.read_csv(paths.sample_submission))[:10]}
P4['status'] = 'PASS' if (P4['deterministic_predictions'] and P4['bundle_paths_ok'] and P4['lightgbm_text_models_loaded'] == 5) else 'FAIL'
print(json.dumps(P4, indent=1, default=str))
REPORT['P4'] = P4
(D['parity'] / 'parity_report.json').write_text(json.dumps(REPORT, indent=2, default=str), encoding='utf-8')
print({k: v.get('status') for k, v in REPORT.items() if isinstance(v, dict)})

{
 "environment": {
  "python": "3.10.20",
  "platform": "Windows-10-10.0.26200-SP0",
  "numpy": "1.26.4",
  "pandas": "2.3.3",
  "pyarrow": "23.0.1",
  "lightgbm": "4.7.0",
  "numba": "unavailable (ModuleNotFoundError)",
  "psutil": "7.2.2"
 },
 "network_disabled_here": false,
 "lightgbm_text_models_loaded": 5,
 "parquet_ok": true,
 "deterministic_predictions": true,
 "bundle_paths_ok": true,
 "submission_problems_unpadded_visible_test": [],
 "status": "PASS"
}
{'P1': 'PASS', 'P1b': 'PASS', 'P2': 'PASS', 'P3': 'PASS', 'P4': 'PASS'}
